In [0]:
%sql
CREATE TABLE if not EXISTS de_learning_workspace.gold.dim_product
(
    product_sk BIGINT GENERATED ALWAYS AS IDENTITY,

    product_id INT,

    product_name STRING,
    category STRING,
    brand STRING,
    price DECIMAL(10,2),
    stock_available INT,

    effective_start_date TIMESTAMP,
    effective_end_date TIMESTAMP,

    is_current BOOLEAN,

    created_timestamp TIMESTAMP,
    updated_timestamp TIMESTAMP
)
USING DELTA;

In [0]:
-- %sql

-- INSERT INTO de_learning_workspace.gold.dim_product
-- (
--     product_id,
--     product_name,
--     category,
--     brand,
--     price,
--     stock_available,

--     effective_start_date,
--     effective_end_date,

--     is_current,

--     created_timestamp,
--     updated_timestamp
-- )

-- SELECT

--     product_id,
--     product_name,
--     category,
--     brand,
--     price,
--     stock_available,

--     current_timestamp(),
--     NULL,

--     TRUE,

--     created_at,
--     updated_at

-- FROM de_learning_workspace.silver.products;

In [0]:
UPDATE de_learning_workspace.gold.dim_product AS target
SET
    is_current = false,
    effective_end_date = current_timestamp()
WHERE target.is_current = true
AND EXISTS
(
    SELECT 1
    FROM de_learning_workspace.silver.products source
    WHERE source.product_id = target.product_id
      AND
      (
             source.product_name <> target.product_name
          OR source.category     <> target.category
          OR source.brand        <> target.brand
          OR source.price        <> target.price
      )
);

In [0]:
INSERT INTO de_learning_workspace.gold.dim_product
(
    product_id,
    product_name,
    category,
    brand,
    price,
    stock_available,
    effective_start_date,
    effective_end_date,
    is_current,
    created_timestamp,
    updated_timestamp
)

SELECT
    s.product_id,
    s.product_name,
    s.category,
    s.brand,
    s.price,
    s.stock_available,
    current_timestamp(),
    NULL,
    TRUE,
    s.created_at,
    s.updated_at

FROM de_learning_workspace.silver.products AS s

WHERE NOT EXISTS
(
    SELECT 1
    FROM de_learning_workspace.gold.dim_product AS t
    WHERE t.product_id = s.product_id
      AND t.is_current = TRUE
);

In [0]:
%python
print("Gold Layer Dim Products Loaded Successfully")